This notebook will demonstrate how to use the **constrained training algorithms** implemented in this toolkit, designed to augment your normal **PyTorch** training routine.

The algorithms implemented in the **humancompatible.train.optim** subpackage share a similar idea.

1. Define your PyTorch optimizer like you would normally. Let's call it`opt`.
2. Define a dual "optimizer" from`humancompatible.train.optim`; it will keep track of and update the dual variables. Let's call it`dual`.
3. In the training loop:
    - Compute the constraints and the objective function (loss);
    - Calculate the Lagrangian and update the dual variables using `dual.forward_update` method;
    - Run a `backward` pass through the Lagrangian (instead of the loss);
    - Run `opt.step`.

Let's train a simple classification model, putting a constraint on the norm of each layer's parameters.

In the canonical form, the algorithm expects equality constraints that are equal to 0; however, we can easily transform arbitrary inequality constraints to that form.

In [ ]:
# load and prepare data

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import numpy as np
from folktables import ACSDataSource, ACSIncome, generate_categories

torch.set_default_dtype(torch.double)

# load folktables data
data_source = ACSDataSource(survey_year="2018", horizon="1-Year", survey="person")
acs_data = data_source.get_data(states=["CA"], download=True)
definition_df = data_source.get_definitions(download=True)
categories = generate_categories(
    features=ACSIncome.features, definition_df=definition_df
)
df_feat, df_labels, _ = ACSIncome.df_to_pandas(
    acs_data, categories=categories, dummies=True
)

sens_cols = ["SEX_Female", "SEX_Male"]
features = df_feat.drop(columns=sens_cols).to_numpy(dtype=float)
groups = df_feat[sens_cols].to_numpy(dtype=float)
labels = df_labels.to_numpy(dtype=float)

# split into train and test
X_train, X_test, y_train, y_test, groups_train, groups_test = train_test_split(
    features, labels, groups, test_size=0.3, random_state=42
)

# scale
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# further split test into val and test
X_val, X_test, y_val, y_test, groups_val, groups_test = train_test_split(
    X_test, y_test, groups_test, test_size=0.5, random_state=42
)

# make into a pytorch dataset, remove the sensitive attribute
features_train = torch.tensor(X_train)
labels_train = torch.tensor(y_train)
sens_train = torch.tensor(groups_train)
dataset_train = torch.utils.data.TensorDataset(features_train, labels_train)
dataset_train_s = torch.utils.data.TensorDataset(features_train, sens_train, labels_train)

# make val and test into pytorch datasets
features_val = torch.tensor(X_val)
labels_val = torch.tensor(y_val)
sens_val = torch.tensor(groups_val)
dataset_val = torch.utils.data.TensorDataset(features_val, labels_val)
dataset_val_s = torch.utils.data.TensorDataset(features_val, sens_val, labels_val)

features_test = torch.tensor(X_test)
labels_test = torch.tensor(y_test)
sens_test = torch.tensor(groups_test)
dataset_test = torch.utils.data.TensorDataset(features_test, labels_test)
dataset_test_s = torch.utils.data.TensorDataset(features_test, sens_test, labels_test)

In [ ]:
import torch
from humancompatible.train.dual_optim import ALM, PBM
from humancompatible.train.dual_optim import MoreauEnvelope
from torch.nn import Sequential
torch.manual_seed(0)

dataloader = torch.utils.data.DataLoader(dataset_train, batch_size=16, shuffle=False)

hsize1 = 12
hsize2 = 12
model = Sequential(
    torch.nn.Linear(features_train.shape[1], hsize1),
    torch.nn.ReLU(),
    torch.nn.Linear(hsize1, hsize2),
    torch.nn.ReLU(),
    torch.nn.Linear(hsize2, 1),
)

optimizer = MoreauEnvelope(
    torch.optim.Adam(model.parameters(), lr=0.01),
)

m = len(list(model.parameters()))
dual = ALM(m=6, lr=0.1, momentum=0.5, dampening=0.5)

constraint_bounds = [1.0] * m
epochs = 4
criterion = torch.nn.BCEWithLogitsLoss()

# do a dummy backward pass
model(dataset_train[0][0]).backward()
model.zero_grad()

In [ ]:
for epoch in range(epochs):
    loss_log = []
    c_log = []
    slack_log = []
    duals_log = []
    for batch_input, batch_label in dataloader:
        # calculate constraints and constraint grads
        c_log.append([])
        constraints = []
        for i, param in enumerate(model.parameters()):
            norm = torch.linalg.norm(param)
            # convert constraint to equality
            norm_viol = torch.max(norm - constraint_bounds[i], torch.zeros(1, requires_grad=True))
            constraints.append(norm_viol)
            c_log[-1].append(norm.detach().numpy())

        constraints = torch.cat(constraints)
        batch_output = model(batch_input)
        bce_loss = criterion(batch_output, batch_label)
        
        lag_loss = dual.forward_update(bce_loss, constraints)
        lag_loss.backward()
        
        optimizer.step()
        optimizer.zero_grad()
        
        loss_log.append(bce_loss.detach().numpy())
        duals_log.append(dual.duals)

    print(
        f"Epoch: {epoch}, "
        f"loss: {loss_log[-1]}, "
        f"constraints: {c_log[-1]}, "
        f"dual: {duals_log[-1]}"
    )

Epoch: 0, loss: 0.47869452804729135, constraints: [array(0.97763408), array(0.99855156), array(0.99447135), array(0.97158111), array(0.9731961), array(0.40546015)], dual: tensor([2.3179, 0.2087, 0.7987, 0.0354, 0.5087, 0.0000])
Epoch: 1, loss: 0.4757059559773724, constraints: [array(0.98706547), array(0.99664963), array(0.97850095), array(0.9687625), array(0.98061239), array(0.42209722)], dual: tensor([3.0719, 0.2849, 0.9189, 0.0389, 0.6584, 0.0000])
Epoch: 2, loss: 0.4737315108750157, constraints: [array(0.95476757), array(0.99733958), array(0.98394266), array(0.96146837), array(0.99426166), array(0.41562445)], dual: tensor([3.7819, 0.3389, 1.0177, 0.0426, 0.7679, 0.0000])
Epoch: 3, loss: 0.48785282681507863, constraints: [array(0.91625595), array(0.99440872), array(0.99530238), array(0.96206378), array(0.99997336), array(0.4009129)], dual: tensor([4.4676, 0.3882, 1.0966, 0.0461, 0.8512, 0.0000])


The model is now trained subject to the constraints we set.

---
---

It is also possible to train subject to **stochastic constraints**. One of the main use-cases for that is **fairness**. Let's now train a fairness-constrained model on the `folktables` dataset.

To track the "fairness" of our model, we will track a **fairness statistic** across groups. This can be the Positive Rate, or PPV, or any other important metric we'd like to keep an eye on. To calculate it, we will use the `fairret` package and its `NormLoss`.
The latter calculates the ratio between the value of a statistic for each group and the overall value: $\sum_{s\in S}{|1-\frac{f(\theta, X_s, y_s)}{f(\theta, X, y)}|}$, where $S$ is the set of groups. 

To make sure each batch contains representatives of each protected group, we can use the BalancedBatchSampler from the `fairness.utils` subpackage - a custom PyTorch `Sampler` which yields an equal number of samples from each subgroup in each batch.

In [ ]:
from fairret.statistic import PositiveRate, TruePositiveRate, FalsePositiveRate, PositivePredictiveValue, FalseOmissionRate
from fairret.loss import NormLoss
from humancompatible.train.fairness.utils import BalancedBatchSampler

batch_size = 128
epochs = 10

fair_sampler = BalancedBatchSampler(group_onehot=sens_train, batch_size=batch_size)
loader = torch.utils.data.DataLoader(dataset_train_s, sampler=fair_sampler)
# loader = torch.utils.data.DataLoader(dataset_train_s, batch_size=batch_size, shuffle=True)

criterion = torch.nn.BCEWithLogitsLoss()
statistic = PositivePredictiveValue()
stat_name = statistic._get_name()
stat_needs_labels = stat_name != 'PositiveRate'
fair_criterion = NormLoss(statistic=statistic)

hsize1 = 32
hsize2 = 16

First, let us train an unconstrained model:

In [ ]:
from torch.optim import Adam
from torch.nn import Sequential

model_uncon = Sequential(
    torch.nn.Linear(features_train.shape[1], hsize1),
    torch.nn.ReLU(),
    torch.nn.Linear(hsize1, hsize2),
    torch.nn.ReLU(),
    torch.nn.Linear(hsize2, 1),
)

optimizer = Adam(model_uncon.parameters())

train_losses = []
train_f = []
val_losses =[]
val_f = []

for epoch in range(epochs):
    
    ### logging ###
    with torch.no_grad():
        logits_train = model_uncon(features_train)
        loss_train = criterion(logits_train, labels_train)
        logits_val = model_uncon(features_val)
        loss_val = criterion(logits_val, labels_val)

        if stat_needs_labels:
            fair_train = fair_criterion(model_uncon(features_train), sens_train, labels_train)
            fair_val = fair_criterion(model_uncon(features_val), sens_val, labels_val)
        else:
            fair_train = fair_criterion(model_uncon(features_train), sens_train)
            fair_val = fair_criterion(model_uncon(features_val), sens_val)
        train_losses.append(loss_train)
        train_f.append(fair_train)
        val_losses.append(loss_val)
        val_f.append(fair_val)
    ###############

    for batch_feat, _, batch_label in loader:
        optimizer.zero_grad()

        logit = model_uncon(batch_feat)
        loss = criterion(logit, batch_label)
        loss.backward()

        optimizer.step()
    
    print(f"Epoch: {epoch}, loss: {loss_train} / {loss_val}, constraint: {fair_train} / {fair_val}")

Epoch: 0, loss: 0.6894747147049205 / 0.6899606496716271, constraint: 0.28573409744738465 / 0.2850625398434343
Epoch: 1, loss: 0.39357662461673376 / 0.4031995891706854, constraint: 0.13457478438191228 / 0.13086227530321326
Epoch: 2, loss: 0.3856550176927553 / 0.39908719563243766, constraint: 0.1298646688941303 / 0.12789216706307927
Epoch: 3, loss: 0.3782571061662329 / 0.395586736778833, constraint: 0.1246901669980246 / 0.12190639144854121
Epoch: 4, loss: 0.3728354132945732 / 0.39436607828790937, constraint: 0.12745675009211244 / 0.12596998404173676
Epoch: 5, loss: 0.3672457183576824 / 0.3942474378885069, constraint: 0.12160979045169418 / 0.12201643719503441
Epoch: 6, loss: 0.36257055931529186 / 0.39395193963663255, constraint: 0.11451340257688813 / 0.11650283741025458
Epoch: 7, loss: 0.35832263637832346 / 0.39439866195144263, constraint: 0.1204040469012082 / 0.12319323637985036
Epoch: 8, loss: 0.35281799753895243 / 0.396124790996196, constraint: 0.1100188134548924 / 0.11369159983044852


In [ ]:
print(f'{stat_name} by group:')
preds_train = torch.nn.functional.sigmoid(model_uncon(features_train))
print(f'Train: {(statistic(preds_train, sens_train, labels_train) if stat_needs_labels else statistic(preds_train, sens_train)).detach().numpy()}')
preds_test = torch.nn.functional.sigmoid(model_uncon(features_test))
print(f'Test: {(statistic(preds_test, sens_test, labels_test) if stat_needs_labels else statistic(preds_test, sens_test)).detach().numpy()}')

PositivePredictiveValue by group:
Train: [0.6898064  0.76749101]
Test: [0.66612221 0.74258116]


In [ ]:
print("Loss:")
print(f'Train: {criterion(model_uncon(features_train), labels_train).detach().numpy()}')
print(f'Test: {criterion(model_uncon(features_test), labels_test).detach().numpy()}')

Loss:
Train: 0.34276154768593603
Test: 0.4043249860669277


---
---

We see that unconstrained optimization leads to a large **discrepancy in the positive rate**. Say we want the fairret term to be no greater than 0.1. We can enforce this constraint with the **constrained** training algorithms.

First, we will try the **Augmented Lagrangian**. Natively, it only works with equality constraints, so we will introduce **slack variables**.

A note: stochastic constrained optimization algorithm benefit greatly from smoothing. We provide the `MoreauEnvelope` class that adds an L2 smoothing term to the model's loss function gradient (without spending any resources during the backward call).

In [ ]:
from torch.nn import Sequential
from torch.optim import Adam
from humancompatible.train.dual_optim import ALM, MoreauEnvelope, PBM

model_con = Sequential(
    torch.nn.Linear(features_train.shape[1], hsize1),
    torch.nn.ReLU(),
    torch.nn.Linear(hsize1, hsize2),
    torch.nn.ReLU(),
    torch.nn.Linear(hsize2, 1),
)

# Define data and optimizers
optimizer = MoreauEnvelope(Adam(model_con.parameters()))
dual = ALM(m=1, lr=0.005, momentum=0.6)

slack_vars = torch.zeros(1, requires_grad=True)
optimizer.add_param_group({"params":slack_vars})

# fairness constraint bound
fair_crit_bound = 0.05

In [ ]:
train_losses = []
train_f = []
val_losses =[]
val_f = []

fllog = []

for epoch in range(epochs):
    
    # logging
    with torch.no_grad():
        logits_train = model_con(features_train)
        loss_train = torch.nn.functional.binary_cross_entropy_with_logits(logits_train, labels_train)
        logits_val = model_con(features_val)
        loss_val = torch.nn.functional.binary_cross_entropy_with_logits(logits_val, labels_val)

        if stat_needs_labels:
            fair_train = fair_criterion(logits_train, sens_train, labels_train)
            fair_val = fair_criterion(logits_val, sens_val, labels_val)
        else:
            fair_train = fair_criterion(logits_train, sens_train)
            fair_val = fair_criterion(logits_val, sens_val)

        lagr_train = dual.forward(loss_train, fair_train.unsqueeze(0))
        train_losses.append(loss_train)
        train_f.append(fair_train)
        val_losses.append(loss_val)
        val_f.append(fair_val)
    

    for batch_input, batch_sens, batch_label in loader:
        # do forward pass
        batch_out = model_con(batch_input)
        loss = criterion(batch_out, batch_label)

        # evaluate fairness criterion (constraint)
        if stat_needs_labels:
            fair_loss = fair_criterion(batch_out.squeeze(0), batch_sens.squeeze(0), batch_label.squeeze(0))
        else:
            fair_loss = fair_criterion(batch_out.squeeze(0), batch_sens.squeeze(0))
        fair_constraint = fair_loss - fair_crit_bound + slack_vars[0]
    
        # calculate lagrangian, update dual variables
        lagrangian = dual.forward_update(loss, fair_constraint.unsqueeze(0))

        # gradient, optimizer step
        lagrangian.backward()
        optimizer.step()
        optimizer.zero_grad()

        # set slacks to be non-negative
        with torch.no_grad():
            for s in slack_vars:
                if s < 0:
                    s.zero_()
    
    print(f"Epoch: {epoch}, loss: {loss_train} / {loss_val}, fairret loss: {fair_train} / {fair_val}, L: {lagr_train}")

Epoch: 0, loss: 0.7035587984033863 / 0.702595964491977, fairret loss: 0.2861890739065852 / 0.28554417331091153, L: 0.7445108914151407
Epoch: 1, loss: 0.4019501759361696 / 0.40860773638971737, fairret loss: 0.08621726080404879 / 0.0866653582203164, L: 0.4537554693687558
Epoch: 2, loss: 0.4033264246334459 / 0.4105001859114869, fairret loss: 0.059132370930958134 / 0.06120498811652164, L: 0.46120777992800416
Epoch: 3, loss: 0.4063307094362159 / 0.41504312107266905, fairret loss: 0.04754146194271103 / 0.053471899055216876, L: 0.47045796948358826
Epoch: 4, loss: 0.4075375260475604 / 0.41928565090347314, fairret loss: 0.045364276769779166 / 0.05078456799010278, L: 0.4860117382109791
Epoch: 5, loss: 0.4081833622086961 / 0.41895843541920663, fairret loss: 0.03756902741094881 / 0.04257730516411329, L: 0.48709773132052014
Epoch: 6, loss: 0.41537670822631584 / 0.42845008380917843, fairret loss: 0.027939050998332937 / 0.03669636935121612, L: 0.4836264637258443
Epoch: 7, loss: 0.4079718202663389 / 0

In [ ]:
print(f'{stat_name} by group:')
preds_train = torch.nn.functional.sigmoid(model_con(features_train))
print(f'Train: {(statistic(preds_train, sens_train, labels_train) if stat_needs_labels else statistic(preds_train, sens_train)).detach().numpy()}')
preds_test = torch.nn.functional.sigmoid(model_con(features_test))
print(f'Test: {(statistic(preds_test, sens_test, labels_test) if stat_needs_labels else statistic(preds_test, sens_test)).detach().numpy()}')

PositivePredictiveValue by group:
Train: [0.7172889  0.73102661]
Test: [0.7034836  0.72550485]


In [ ]:
print('Fairness criterion value:')
print(f'Train: {fair_criterion(model_con(features_train), sens_train, labels_train).detach().numpy()}')
print(f'Test: {fair_criterion(model_con(features_test), sens_test, labels_test).detach().numpy()}')

Fairness criterion value:
Train: 0.018932754279296193
Test: 0.0307223473440027


In [ ]:
print("Loss:")
print(f'Train: {criterion(model_con(features_train), labels_train).detach().numpy()}')
print(f'Test: {criterion(model_con(features_test), labels_test).detach().numpy()}')

Loss:
Train: 0.4203671444221577
Test: 0.43931647484861047


Curoiusly, in this case, the Augmented Lagrangian converges to a feasible local minimum not on the constraint boundary (our constraint is strictly less than the bound we set for it).

***
Now, let us try the **Penalty-Barrier** method. In contrast to the ALM, it can handle inequality constraints natively, so we have no need for slacks.

In [ ]:
from torch.nn import Sequential
from torch.optim import Adam
from humancompatible.train.dual_optim import ALM, MoreauEnvelope, PBM

model_con = Sequential(
    torch.nn.Linear(features_train.shape[1], hsize1),
    torch.nn.ReLU(),
    torch.nn.Linear(hsize1, hsize2),
    torch.nn.ReLU(),
    torch.nn.Linear(hsize2, 1),
)

# Define data and optimizers
optimizer = MoreauEnvelope(Adam(model_con.parameters()))

dual = PBM(
    m=1,
    penalty_update='dimin_adapt',
    penalty_mult=0.7,
    gamma=0.7,
    init_duals=0.001,
    init_penalties=10.,
    penalty_range=(0.01, 1000.),
    dual_range=(0.01, 100.),
    delta=0.7,
)

fair_crit_bound = 0.05
epochs = 20

In [ ]:
train_losses = []
train_f = []
val_losses =[]
val_f = []

p_log = []
lambda_log = []

for epoch in range(epochs):
    
    # logging
    with torch.no_grad():
        logits_train = model_con(features_train)
        loss_train = torch.nn.functional.binary_cross_entropy_with_logits(logits_train, labels_train)
        logits_val = model_con(features_val)
        loss_val = torch.nn.functional.binary_cross_entropy_with_logits(logits_val, labels_val)

        if stat_needs_labels:
            fair_train = fair_criterion(logits_train, sens_train, labels_train)
            fair_val = fair_criterion(logits_val, sens_val, labels_val)
        else:
            fair_train = fair_criterion(logits_train, sens_train)
            fair_val = fair_criterion(logits_val, sens_val)

        lagr_train = dual.forward(loss_train, fair_train.unsqueeze(0))
        train_losses.append(loss_train)
        train_f.append(fair_train)
        val_losses.append(loss_val)
        val_f.append(fair_val)

    for batch_input, batch_sens, batch_label in loader:
 
        batch_out = model_con(batch_input)
        loss = criterion(batch_out, batch_label)

        fair_loss = fair_criterion(batch_out.squeeze(0), batch_sens.squeeze(0), batch_label.squeeze(0))
            
        fair_constraint = fair_loss - fair_crit_bound

        lagrangian = dual.forward_update(loss, fair_constraint.unsqueeze(0))
        p_log.append(dual.penalties)
        lambda_log.append(dual.duals)
        lagrangian.backward()
        optimizer.step()
        optimizer.zero_grad()
    
    print(f"Epoch: {epoch}, loss: {loss_train} / {loss_val}, fairret loss: {fair_train} / {fair_val}, L: {lagr_train}")

Epoch: 0, loss: 0.6831327336947913 / 0.6847378109485782, fairret loss: 0.28778091587725796 / 0.2864476858167291, L: 0.6834246555034458
Epoch: 1, loss: 0.39477076690123436 / 0.40304701144594823, fairret loss: 0.09913512326095097 / 0.09880546912054422, L: 0.41966946586149767
Epoch: 2, loss: 0.3903033699924333 / 0.40320959762010966, fairret loss: 0.08166434778098064 / 0.08395577700473322, L: 0.42318244310167463
Epoch: 3, loss: 0.3866629069830693 / 0.4008118989293931, fairret loss: 0.07235167610829474 / 0.07673964744056727, L: 0.41827096832463934
Epoch: 4, loss: 0.3811744049839665 / 0.39856562314798416, fairret loss: 0.07781276360238876 / 0.08319638146068964, L: 0.4160584359744143
Epoch: 5, loss: 0.3788702247102069 / 0.39873228876677735, fairret loss: 0.06829656838195586 / 0.07514906384832687, L: 0.41024871116243306
Epoch: 6, loss: 0.3751530106515541 / 0.3987976293602634, fairret loss: 0.07010293041060489 / 0.07657215270087192, L: 0.40811798917556286
Epoch: 7, loss: 0.37173894222466225 / 0

In [ ]:
print(f'{stat_name} by group:')
preds_train = torch.nn.functional.sigmoid(model_con(features_train))
print(f'Train: {(statistic(preds_train, sens_train, labels_train) if stat_needs_labels else statistic(preds_train, sens_train)).detach().numpy()}')
preds_test = torch.nn.functional.sigmoid(model_con(features_test))
print(f'Test: {(statistic(preds_test, sens_test, labels_test) if stat_needs_labels else statistic(preds_test, sens_test)).detach().numpy()}')

PositivePredictiveValue by group:
Train: [0.71272831 0.75038821]
Test: [0.67337277 0.72768037]


In [ ]:
print('Fairness criterion value:')
print(f'Train: {fair_criterion(model_con(features_train), sens_train, labels_train).detach().numpy()}')
print(f'Test: {fair_criterion(model_con(features_test), sens_test, labels_test).detach().numpy()}')

Fairness criterion value:
Train: 0.051235441385318214
Test: 0.07700492878212994


In [ ]:
print("Loss:")
print(f'Train: {criterion(model_con(features_train), labels_train).detach().numpy()}')
print(f'Test: {criterion(model_con(features_test), labels_test).detach().numpy()}')

Loss:
Train: 0.33969423673233096
Test: 0.4168362095969089
